# Hyperparamter tuning

- Mean : None
- KNN : Tuning k 
- MICE : None
- SoftImpute : Tuning shrinking value (lambda) 
- HyperImpute : None
- DiffPuter : Tuning hidden dimensional
- GRAPE : Tuning epochs and capacity

In [ ]:
# Path setup — run this first. Anchors all paths to the repo root so the
# notebook works regardless of the directory it's launched from.
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk upward until we find a dir containing both 'src' and 'data'."""
    for parent in [start, *start.parents]:
        if (parent / "src").is_dir() and (parent / "data").is_dir():
            return parent
    # Fallback: assume the notebook lives in <root>/notebooks/
    return start.parent

# Path.cwd() is where Jupyter was launched; __file__ isn't defined in nbs.
REPO_ROOT = _find_repo_root(Path.cwd())

SRC = REPO_ROOT / "src"
RESULTS_DIR = REPO_ROOT / "results"
TUNED_PARAMS_PATH = RESULTS_DIR / "tuned_params.json"
DATA_DIR    = REPO_ROOT / "data" / "Scenario5"
FIGURES_DIR = RESULTS_DIR / "figures"
CSV_DIR     = RESULTS_DIR / "csv"
for _d in (FIGURES_DIR, CSV_DIR):
    _d.mkdir(parents=True, exist_ok=True)

_diffputer = REPO_ROOT / "external" / "DiffPuter"
for _p in (
    REPO_ROOT,                             
    SRC,
    _diffputer / "baselines" / "GRAPE",
    _diffputer / "baselines",
    _diffputer,
):
    _sp = str(_p.resolve())
    if _p.exists() and _sp not in sys.path:
        sys.path.insert(0, _sp)

print("Tuned params will be saved to:", TUNED_PARAMS_PATH)

# Imports
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from imputers.imputers import (
    KNNImputerWrapper, 
    SoftImputeWrapper
)
from imputers.diffputer_imputer import DiffPuterImputer
from imputers.grape_imputer import GRAPEImputer

from common.experiment import ExperimentConfig
 
from common.tuning import (
    tune_imputer, tune_knn, tune_softimpute,
    softimpute_shrinkage_grid, save_tuned_params, load_tuned_params,
)

from common.scenario5.scenario5_data import load_scenario5_clean

clean_data = load_scenario5_clean(DATA_DIR)

DATASET = "scenario5"

Tuned params will be saved to: /mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/results/tuned_params.json
Parsing location files...
Parsing mmWave power files...
Argmax matches unit1_beam_index: 100.00%
Clean data shape: (2300, 70)
Missing values after parsing: 0


In [2]:
# Configuration for tuning
BEAM_COLS = [f"beam_{i:02d}" for i in range(64)]
GPS_COLS = [
    "unit2_lat",
    "unit2_lon",
    "unit2_direction",
    "unit2_num_sat",
    "unit2_PDOP",
    "unit2_HDOP",
]
RETAINED_COLS = BEAM_COLS + GPS_COLS

cfg = ExperimentConfig(
    target_cols=BEAM_COLS,                  
    retained_cols=BEAM_COLS + GPS_COLS,     
    mar_driver_cols=["unit2_HDOP"],
    mar_score_mode="single",
    proportions=[0.10, 0.30, 0.50],
    n_seeds=1,
)

In [3]:
# Tune kNN: n_neighbors

knn_result = tune_knn(
    clean_data, cfg, KNNImputerWrapper,
    grid=(3, 5, 10, 20, 50),
    prop=0.30,
    tuning_seeds=(0, 1, 2),
    verbose=True,
)
knn_result["table"]

  n_neighbors=3: RMSE 0.3058 ± 0.0034
  n_neighbors=5: RMSE 0.2988 ± 0.0051
  n_neighbors=10: RMSE 0.3020 ± 0.0044
  n_neighbors=20: RMSE 0.3194 ± 0.0053
  n_neighbors=50: RMSE 0.3770 ± 0.0044
  -> best n_neighbors = 5.0 (RMSE 0.2988)


,n_neighbors,rmse_mean,rmse_std
0,5,0.298789,0.005116
1,10,0.301952,0.004442
2,3,0.305756,0.003423
3,20,0.319389,0.005320
4,50,0.377004,0.004444


In [4]:
# Tune SoftImpute: shrinkage_value (regularization lambda)

grid = softimpute_shrinkage_grid(clean_data, cfg)
soft_result = tune_softimpute(clean_data, cfg, SoftImputeWrapper,
                              grid=grid, tuning_seeds=(0, 1, 2))
display(soft_result["table"])

soft_best = soft_result["best_value"]
softimpute_shrinkage = None if (soft_best is None or pd.isna(soft_best)) else float(soft_best)
print("Selected softimpute_shrinkage =", softimpute_shrinkage)

max singular value = 182.954 (fancyimpute default shrinkage = 3.6591)
  shrinkage_value=None: RMSE 0.2467 ± 0.0042
  shrinkage_value=1.8295: RMSE 0.2492 ± 0.0042
  shrinkage_value=3.6591: RMSE 0.2474 ± 0.0040
  shrinkage_value=9.1477: RMSE 0.2995 ± 0.0039
  shrinkage_value=18.2954: RMSE 0.4058 ± 0.0037
  -> best shrinkage_value = nan (RMSE 0.2467)


,shrinkage_value,rmse_mean,rmse_std
0,NaN,0.246705,0.004224
1,3.6591,0.247386,0.003953
2,1.8295,0.249154,0.004247
3,9.1477,0.299503,0.003861
4,18.2954,0.405836,0.003694


Selected softimpute_shrinkage = None


In [5]:
# Tune DiffPuter: hid_dim (hidden dimension size) 
diff_result = tune_imputer(
    clean_data, cfg,
    factory=lambda h: DiffPuterImputer(
        hid_dim=h,
        n_em_iterations=2,
        n_train_epochs=3000,
        early_stopping_patience=300,
        verbose=False,
    ),
    param_grid=[128, 256, 512, 1024],
    param_name="hid_dim",
    prop=0.30,
    tuning_seeds=(0,),          
    verbose=True,
)
display(diff_result["table"])

DIFFPUTER_HID_DIM = int(diff_result["best_value"])

# Tie check: if the top two overlap within error bars, prefer the smaller.
ranked = diff_result["table"].sort_values("rmse_mean").reset_index(drop=True)
if len(ranked) >= 2:
    top, second = ranked.iloc[0], ranked.iloc[1]
    top_std = 0.0 if pd.isna(top["rmse_std"]) else top["rmse_std"]
    sec_std = 0.0 if pd.isna(second["rmse_std"]) else second["rmse_std"]
    if not (top["rmse_mean"] + top_std < second["rmse_mean"] - sec_std):
        DIFFPUTER_HID_DIM = int(min(top["hid_dim"], second["hid_dim"]))
        print(f"Top two overlap -> preferring smaller hid_dim={DIFFPUTER_HID_DIM}")
print("Selected diffputer_hid_dim =", DIFFPUTER_HID_DIM)

KeyboardInterrupt: 

In [ ]:
# Tune GRAPE: epochs and capacity (node/edge dim)

# 1) epochs sweep (capacity at default node/edge dim = 64)
grape_epochs_result = tune_imputer(
    clean_data, cfg,
    factory=lambda e: GRAPEImputer(epochs=e),
    param_grid=[2000, 5000, 10000, 20000],
    param_name="epochs",
    prop=0.30,
    tuning_seeds=(0,),
    verbose=True,
)
display(grape_epochs_result["table"])
GRAPE_EPOCHS = int(grape_epochs_result["best_value"])
print("Best GRAPE epochs =", GRAPE_EPOCHS)

# 2) capacity sweep (epochs fixed at the winner above)
grape_capacity_result = tune_imputer(
    clean_data, cfg,
    factory=lambda d: GRAPEImputer(epochs=GRAPE_EPOCHS, node_dim=d, edge_dim=d),
    param_grid=[32, 64, 128],
    param_name="node_edge_dim",
    prop=0.30,
    tuning_seeds=(0,),
    verbose=True,
)
display(grape_capacity_result["table"])
GRAPE_CAPACITY = int(grape_capacity_result["best_value"])
print("Best GRAPE node/edge dim =", GRAPE_CAPACITY)

In [ ]:
# Saving parameters (For JSON compatibility)
softimpute_shrinkage = soft_result["best_value"]
if softimpute_shrinkage is None or pd.isna(softimpute_shrinkage):
    softimpute_shrinkage = None
else:
    softimpute_shrinkage = float(softimpute_shrinkage)

save_tuned_params(TUNED_PARAMS_PATH, DATASET, {
    "knn_n_neighbors":      int(knn_result["best_value"]),
    "softimpute_shrinkage": softimpute_shrinkage,       
    "diffputer_hid_dim":    int(DIFFPUTER_HID_DIM),
    "grape_epochs":        int(GRAPE_EPOCHS),
    "grape_node_edge_dim": int(GRAPE_CAPACITY),
})

print("Saved for", DATASET, "->", load_tuned_params(TUNED_PARAMS_PATH, DATASET))